# 02 · Our method: LLM candidates + A* search

**Mini-crossword solver** (Avery Mattoon, Yutong Hu). CS 557 final project.

How it works (code: `src/crossword/astar.py`):
1. **One LLM call per puzzle** asks for the top-k answers to every clue, each with a confidence 0–100.
2. **A\* search** fills the grid. Placing a word costs −log(confidence), so the cheapest full grid is the one the LLM finds most likely overall. The heuristic adds up the cheapest still-fitting candidate for each empty slot, so it never overestimates (admissible). Candidates that no longer fit the placed letters are dropped (forward checking), and the slot with the fewest candidates left is filled next.
3. If no candidate fits a slot, the slot can stay **unresolved** at a penalty, and its letters come from the crossing words. We then **re-prompt** the LLM for that slot with the letter pattern (e.g. `S _ A _ E`) and search again.

ToT asks the LLM at every step (~166 GPT-4 calls per puzzle). We ask once, plus a few re-prompts.

**Before running:** the same Colab secrets as notebook 01 (`GITHUB_TOKEN`, and `GEMINI_API_KEY` for real runs).

## 0 · Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YutongHu-Natalie/mini_crossword_solver.git"
IN_COLAB = "google.colab" in sys.modules

def secret(name):
    """Colab secret, else environment variable / .env file."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

if IN_COLAB:
    # Clone the repo the first time; afterwards pull, so you always run the latest code.
    token = secret("GITHUB_TOKEN")
    print("GitHub token found:", bool(token))
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    if Path("/content/repo/.git").exists():
        # use the CURRENT token (an old clone may have a stale one saved)
        subprocess.run(["git", "-C", "/content/repo", "remote", "set-url", "origin", url])
        cmd = ["git", "-C", "/content/repo", "pull", "-q"]
    else:
        subprocess.run(["rm", "-rf", "/content/repo"])
        cmd = ["git", "clone", "-q", url, "/content/repo"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = r.stderr.replace(token, "***") if token else r.stderr
        raise RuntimeError(f"git failed:\n{msg}\nCheck the GITHUB_TOKEN secret (classic token, 'repo' "
                           "scope, notebook access on) and that you accepted the repo invite.")
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=True)
else:
    # running locally from notebooks/: work from the repo root
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    try:
        from dotenv import load_dotenv; load_dotenv()   # reads OPENAI_API_KEY etc. from .env
    except ImportError:
        pass

sys.path.insert(0, str(Path("src").resolve()))
for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY", "OPENAI_API_KEY"):
    if secret(name):
        os.environ[name] = secret(name)
if not os.environ.get("GEMINI_API_KEY") and os.environ.get("GOOGLE_API_KEY"):
    os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]   # Colab's "import key" button uses this name

# Optional: keep the LLM cache on Google Drive so it survives Colab restarts.
# from google.colab import drive; drive.mount("/content/drive")
# os.environ["CROSSWORD_CACHE"] = "/content/drive/MyDrive/crossword/llm_cache.sqlite"

print("repo root:", Path.cwd())
print("Gemini key found:", bool(os.environ.get("GEMINI_API_KEY")),
      "| OpenAI key found:", bool(os.environ.get("OPENAI_API_KEY")))

## 1 · Choose the model

* `"gemini"`: free Gemini Flash (needs `GEMINI_API_KEY`). The main model.
* `"qwen"`: Qwen2.5-7B-Instruct on the Colab GPU (Runtime → Change runtime type → GPU). Free; the small-model comparison.
* `"dry"`: a fake model that knows the answers but sometimes leaves them out. Free and instant; checks that everything runs. **Its scores mean nothing.**

In [ ]:
MODEL_CHOICE = "dry"   # "gemini", "qwen" or "dry"

GEMINI_MODEL = "gemini-3.8-flash"     # if this errors or is slow, try a "Flash-Lite" model from AI Studio (higher free limits)
QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"

import pandas as pd
from crossword import get_split, SLOTS
from crossword.llm import make_llm
from crossword.fake import fake_crossword_llm
from crossword.astar import AStarSolver, AStarConfig
from crossword.experiment import run, results_path, load_results, summarize, tot_paper_records

if MODEL_CHOICE == "gemini":
    assert os.environ.get("GEMINI_API_KEY"), "add the GEMINI_API_KEY secret and re-run the setup cell"
    llm = make_llm("gemini", GEMINI_MODEL, request_kwargs={"reasoning_effort": "low"})
elif MODEL_CHOICE == "qwen":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=True)
    llm = make_llm("hf", QWEN_MODEL)
else:
    llm = fake_crossword_llm(miss_rate=0.2)
print("model:", llm.model)

CONFIG = AStarConfig()   # defaults: k=5 candidates per clue, up to 10 re-prompts per puzzle
CONFIG

## 2 · Walk through one puzzle

A dev puzzle, so we never peek at test puzzles while developing. First the candidates the LLM proposed, then the search. The ✓ marks the right answer.

In [ ]:
puzzle = get_split("dev")[0]
solver = AStarSolver(llm, CONFIG, verbose=True)

cands = solver.propose(puzzle)
pd.DataFrame([{
    "slot": s, "clue": puzzle.clue(s), "answer": a,
    "candidates (confidence)": ", ".join(f"{w}{' ✓' if w == a else ''} ({p:.2f})"
                                          for w, p in sorted(cands[s].items(), key=lambda x: -x[1])),
} for s, a in zip(SLOTS, puzzle.answers)]).set_index("slot")

In [ ]:
rec = solver.solve(puzzle)

def show(board):
    return "\n".join(" ".join(board[i*5:(i+1)*5]) for i in range(5))

print("\nours:            gold:")
for a, b in zip(show(rec["board"]).split("\n"), show("".join(puzzle.solution)).split("\n")):
    print(f"{a}       {b}")
print("\nscores:", rec["scores"], "| complete grid:", rec["complete"])
print("LLM calls:", rec["usage"]["calls"], "| tokens:", rec["usage"]["prompt_tokens"] + rec["usage"]["completion_tokens"],
      "| nodes expanded:", rec["nodes"])
print("right answers available: first call", rec["gold_in_initial"], "/ 10, after re-prompts", rec["gold_in_candidates"], "/ 10")

## 3 · Dev run

Run on dev puzzles to check the method and tune settings (k, penalty, re-prompt budget) **before** touching the test set. Results are saved after each puzzle to `results/astar__<model>__dev.jsonl`, so an interrupted run resumes.

The last two columns separate the two ways to fail: the right answer never proposed (an LLM problem), or proposed but not chosen (a search problem).

In [ ]:
N_DEV = 5   # start small on Gemini's free tier; raise it once you see how fast calls go

dev_path = results_path("astar", llm.model, "dev")
dev = run(AStarSolver(llm, CONFIG), get_split("dev")[:N_DEV], dev_path)

def diagnose(records):
    d = summarize(records)
    n = len(records)
    d["complete_grids"] = sum(r["complete"] for r in records)
    d["gold_proposed_first_call_%"] = round(100 * sum(r["gold_in_initial"] for r in records) / (10 * n), 1)
    d["gold_available_after_reprompts_%"] = round(100 * sum(r["gold_in_candidates"] for r in records) / (10 * n), 1)
    return d

pd.Series(diagnose(dev))

## 4 · Test run and comparison with ToT

The same 20 test puzzles ToT reports on. Set `RUN_TEST = True` once you are happy with the dev results. Don't tune on test.

ToT's row comes from the paper's logs (notebook 01). Its cost is GPT-4 at list price, so compare **calls and tokens** across models, not dollars.

In [ ]:
RUN_TEST = False

test_path = results_path("astar", llm.model, "test")
test = run(AStarSolver(llm, CONFIG), get_split("test"), test_path) if RUN_TEST else load_results(test_path)
print(len(test), "test puzzles in", test_path.name)

rows = [{"method": "ToT + GPT-4 (paper)", **summarize(tot_paper_records(prune=True, with_usage=True))}]
if test:
    rows.append({"method": f"Ours (A*) + {test[0]['model']}", **diagnose(test)})
cols = ["n", "letter_%", "word_%", "game_%", "games_solved", "llm_calls/puzzle",
        "input_tokens/puzzle", "output_tokens/puzzle", "nodes/puzzle", "complete_grids",
        "gold_proposed_first_call_%", "gold_available_after_reprompts_%"]
pd.DataFrame(rows).set_index("method").reindex(columns=cols)

## 5 · Share results

Push the new `results/` files so your teammate gets them. Fill in your name and email, set `PUSH = True`, and run. To save this notebook itself (with outputs), use **File → Save a copy in GitHub**.

In [ ]:
PUSH = False
GIT_NAME, GIT_EMAIL = "Your Name", "you@example.com"

if PUSH and IN_COLAB:
    for cmd in (["git", "config", "user.name", GIT_NAME], ["git", "config", "user.email", GIT_EMAIL],
                ["git", "pull", "-q", "--rebase", "--autostash"], ["git", "add", "results/"],
                ["git", "commit", "-q", "-m", f"Add A* results ({llm.model})"],
                ["git", "push", "-q"]):
        print(" ".join(cmd[:3]), "->", subprocess.run(cmd).returncode)